# Day 12 — Aggregations + GROUP BY + HAVING
**Estimated time:** 45–60 min
**Dataset:** `cost_center_actuals.csv`

## Learning Objectives
- Master SUM, COUNT, AVG, MIN, MAX, COUNT DISTINCT in aggregation queries
- Understand WHY `HAVING` exists: it filters on aggregated values, not raw rows
- Know when ROLLUP would apply and why SQLite lacks it (OLAP context)

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

from analytics_bootcamp.config import RAW_DATA_DIR as DATA_DIR
# Load all CSVs
inv   = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
sales = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")
cc    = pd.read_csv(f"{DATA_DIR}/cost_center_actuals.csv")
hr    = pd.read_csv(f"{DATA_DIR}/hr_headcount.csv")
bw    = pd.read_csv(f"{DATA_DIR}/bw_sales_kpis.csv")

# Normalize column names
for df in [inv, sales, cc, hr, bw]:
    df.columns = [c.strip().upper() for c in df.columns]

# In-memory DuckDB — register pandas DataFrames as tables
con = duckdb.connect()
con.register("MATERIALS",   inv)
con.register("SALES",       sales)
con.register("COST_CENTER", cc)
con.register("HR",          hr)
con.register("BW_SALES",    bw)

def q(sql):
    return con.sql(sql).df()

# Verify
for tbl, df in [("MATERIALS",inv),("SALES",sales),("COST_CENTER",cc),("HR",hr),("BW_SALES",bw)]:
    n = q(f"SELECT COUNT(*) AS n FROM {tbl}").iloc[0,0]
    print(f"  {tbl:15s}: {n:,} rows")


In [ ]:
# -- Basic aggregation: BUKRS x GJAHR --
result = q(
    """
    SELECT
        BUKRS,
        GJAHR,
        SUM(ACTUAL_AMT)       AS total_actual,
        SUM(PLAN_AMT)         AS total_plan,
        COUNT(*)              AS record_count,
        COUNT(DISTINCT KOSTL) AS n_cost_centers,
        AVG(ACTUAL_AMT)       AS avg_actual,
        MIN(ACTUAL_AMT)       AS min_actual,
        MAX(ACTUAL_AMT)       AS max_actual
    FROM COST_CENTER
    GROUP BY BUKRS, GJAHR
    ORDER BY GJAHR DESC, total_actual DESC
    """
)
display(result)

In [ ]:
# -- GROUP BY multiple columns with variance calc --
result = q(
    """
    SELECT
        KOSTL,
        KTEXT,
        GJAHR,
        SUM(ACTUAL_AMT)   AS total_actual,
        SUM(PLAN_AMT)     AS total_plan,
        SUM(ACTUAL_AMT) - SUM(PLAN_AMT) AS variance,
        ROUND(
            (SUM(ACTUAL_AMT) - SUM(PLAN_AMT)) * 100.0
            / NULLIF(SUM(PLAN_AMT), 0)
        , 2) AS variance_pct
    FROM COST_CENTER
    GROUP BY KOSTL, KTEXT, GJAHR
    ORDER BY variance_pct DESC
    LIMIT 20
    """
)
display(result)

In [ ]:
# -- HAVING vs WHERE: critical distinction --
# WHERE filters ROWS before aggregation (cannot reference aggregate aliases)
# HAVING filters GROUPS after aggregation
#
# This would FAIL:
#   SELECT KOSTL, SUM(ACTUAL_AMT) AS total
#   FROM COST_CENTER WHERE total > 100000  <- ERROR: 'total' not yet computed
#
# CORRECT pattern:
result = q(
    """
    SELECT
        KOSTL,
        KTEXT,
        GJAHR,
        SUM(ACTUAL_AMT) AS total_actual,
        SUM(PLAN_AMT)   AS total_plan,
        ROUND(
            (SUM(ACTUAL_AMT) - SUM(PLAN_AMT)) * 100.0
            / NULLIF(SUM(PLAN_AMT), 0)
        , 2) AS variance_pct
    FROM COST_CENTER
    WHERE GJAHR = (SELECT MAX(GJAHR) FROM COST_CENTER)
    GROUP BY KOSTL, KTEXT, GJAHR
    HAVING variance_pct > 10
    ORDER BY variance_pct DESC
    """
)
print(f"Cost centers over budget by >10%: {len(result)}")
display(result)

In [ ]:
# -- HAVING with multiple conditions --
result = q(
    """
    SELECT
        KOSTL,
        KTEXT,
        COUNT(DISTINCT PERIOD) AS active_periods,
        SUM(ACTUAL_AMT)        AS total_actual,
        AVG(ACTUAL_AMT)        AS avg_monthly_actual
    FROM COST_CENTER
    WHERE GJAHR = (SELECT MAX(GJAHR) FROM COST_CENTER)
    GROUP BY KOSTL, KTEXT
    HAVING active_periods >= 3
       AND total_actual > 50000
    ORDER BY total_actual DESC
    """
)
display(result)

In [ ]:
# -- COUNT DISTINCT vs COUNT --
result = q(
    """
    SELECT
        VKORG,
        COUNT(*)               AS total_line_items,
        COUNT(DISTINCT VBELN)  AS distinct_orders,
        COUNT(DISTINCT KUNNR)  AS distinct_customers,
        COUNT(DISTINCT MATNR)  AS distinct_materials,
        SUM(NETWR)             AS total_revenue
    FROM SALES
    GROUP BY VKORG
    ORDER BY total_revenue DESC
    """
)
display(result)

In [ ]:
# -- ROLLUP concept: not supported in SQLite --
# In HANA, PostgreSQL, Snowflake:
#   GROUP BY ROLLUP(BUKRS, GJAHR) produces:
#   - one row per BUKRS + GJAHR combination
#   - one subtotal per BUKRS (GJAHR = NULL)
#   - one grand total (both NULL)
#
# SQLite workaround: UNION ALL each aggregation level
result = q(
    """
    SELECT 'Detail'       AS level, BUKRS, CAST(GJAHR AS TEXT) AS GJAHR, SUM(ACTUAL_AMT) AS total
    FROM COST_CENTER GROUP BY BUKRS, GJAHR
    UNION ALL
    SELECT 'BUKRS Total',  BUKRS, 'ALL', SUM(ACTUAL_AMT)
    FROM COST_CENTER GROUP BY BUKRS
    UNION ALL
    SELECT 'Grand Total', 'ALL', 'ALL', SUM(ACTUAL_AMT)
    FROM COST_CENTER
    ORDER BY BUKRS, GJAHR
    """
)
display(result)

---
## Daily Prompt

**Find all cost centers where actual spend exceeded plan by more than 15% in ANY period of fiscal year 2024.**

```sql
-- YOUR SOLUTION
SELECT
    KOSTL,
    KTEXT,
    PERIOD,
    ACTUAL_AMT,
    PLAN_AMT,
    ROUND((ACTUAL_AMT - PLAN_AMT) * 100.0 / NULLIF(PLAN_AMT, 0), 2) AS period_variance_pct
FROM COST_CENTER
WHERE GJAHR = 2024
  -- YOUR SOLUTION: filter for >15% overspend
ORDER BY period_variance_pct DESC
```

> **Hint:** This uses a WHERE clause on raw row-level data, NOT HAVING — you are checking per-period overspend, not an aggregated result.
> Condition: `(ACTUAL_AMT - PLAN_AMT) * 100.0 / NULLIF(PLAN_AMT, 0) > 15`
> Follow-up: `SELECT COUNT(DISTINCT KOSTL) FROM (...) — how many distinct cost centers appear?`